<a href="https://colab.research.google.com/github/yaswanthreddy3-dev/aiplanner_try1/blob/main/POC_AudioTranslator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup: Install necessary libraries

We'll install `faster-whisper` for efficient ASR, `transformers` for NLLB-200 (translation) and T5 (summarization), `accelerate` and `optimum` for performance, `pyannote.audio` for speaker diarization, and `sentencepiece` which is a dependency for some `transformers` models.

In [1]:
%%capture
!pip install transformers accelerate optimum sentencepiece numpy
!pip install faster-whisper pyannote.audio

## 1. Speech-to-Text (ASR) with Faster-Whisper

Faster-Whisper is an optimized version of OpenAI's Whisper model, providing faster inference and lower memory usage. We'll download a sample audio file and then transcribe it.

In [2]:
from faster_whisper import WhisperModel
import os

# Download a sample audio file (e.g., a short English speech)
# In a real application, this would be your live audio stream
# Using a more reliable URL for a sample audio file
!wget -O sample_audio.wav https://github.com/ggerganov/whisper.cpp/raw/master/samples/jfk.wav

# Choose a model size (e.g., 'base' or 'small'). 'large-v2' is more accurate but requires more resources.
# 'medium' is a good balance.
model_size = "medium"

# Run on GPU if available, otherwise CPU. Changed to CPU due to CUDA driver error.
# If you encounter CUDA memory issues, try 'compute_type="int8"' or a smaller model_size.
# If you resolve the CUDA driver issue, you can change device back to "cuda".
model = WhisperModel(model_size, device="cpu", compute_type="int8")
# For GPU (if driver issue is resolved): model = WhisperModel(model_size, device="cuda", compute_type="float16")

segments, info = model.transcribe("sample_audio.wav", beam_size=5)

print(f"Detected language: {info.language} with probability {info.language_probability:.2f}")

transcript = ""
for segment in segments:
    print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")
    transcript += segment.text + " "

print(f"\nFull Transcript:\n{transcript.strip()}")

# Store the transcript for later use
english_transcript = transcript.strip()

--2026-03-28 08:03:23--  https://github.com/ggerganov/whisper.cpp/raw/master/samples/jfk.wav
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ggml-org/whisper.cpp/raw/master/samples/jfk.wav [following]
--2026-03-28 08:03:23--  https://github.com/ggml-org/whisper.cpp/raw/master/samples/jfk.wav
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ggml-org/whisper.cpp/master/samples/jfk.wav [following]
--2026-03-28 08:03:23--  https://raw.githubusercontent.com/ggml-org/whisper.cpp/master/samples/jfk.wav
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, 

## 2. Machine Translation (NMT) with NLLB-200

NLLB-200 (No Language Left Behind) by Meta is a powerful translation model supporting 200+ languages. We will use the `transformers` library to load and use it.

For Japanese to English, the model identifiers would be `nllb-200-distilled-600M` or a larger one like `nllb-200-1.3B` for better accuracy if resources allow. I'll use the `distilled-600M` version for faster execution.

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the tokenizer and model for NLLB-200
model_name = "facebook/nllb-200-distilled-600M"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU if available
if hasattr(model, 'to'):
    try:
        model = model.to("cuda")
        device = "cuda"
    except RuntimeError:
        print("CUDA not available, running on CPU.")
        device = "cpu"
else:
    device = "cpu"


# Example translation: English to Japanese
english_text = english_transcript # Using the transcript from ASR

# Tokenize English text for translation to Japanese
tokenizer.src_lang = "eng_Latn"
encoded_english = tokenizer(english_text, return_tensors="pt").to(device)

# Generate Japanese translation
generated_tokens = model.generate(**encoded_english, forced_bos_token_id=tokenizer.convert_tokens_to_ids("jpn_Jpan"))
japanese_translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
print(f"\nEnglish to Japanese Translation:\n{japanese_translation}")

# Now, let's demonstrate Japanese to English translation
sample_japanese_text = "こんにちは。今日の会議の議題は何ですか？"

# Tokenize Japanese text for translation to English
tokenizer.src_lang = "jpn_Jpan"
encoded_japanese = tokenizer(sample_japanese_text, return_tensors="pt").to(device)

# Generate English translation
generated_tokens_jp_en = model.generate(**encoded_japanese, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"))
english_from_japanese = tokenizer.batch_decode(generated_tokens_jp_en, skip_special_tokens=True)[0]
print(f"\nJapanese to English Translation:\n{english_from_japanese}")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

CUDA not available, running on CPU.

English to Japanese Translation:
だから,アメリカ人の皆さん,自分の国が為する事を聞かないで,自分のために何ができるかを聞かれましょう.

Japanese to English Translation:
Hello, what is the topic of today's meeting?


## 3. Summarization with T5 (Text-to-Text Transfer Transformer)

T5 is a versatile model for various text-to-text tasks, including summarization. We'll use `t5-small` for a quick demonstration, but `t5-base` or `t5-large` would provide better summarization quality.

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the T5 tokenizer and model
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU if available, otherwise CPU
if hasattr(model, 'to'):
    try:
        model = model.to("cuda")
        device = "cuda"
    except RuntimeError:
        print("CUDA not available, running on CPU.")
        device = "cpu"
else:
    device = "cpu"

# Let's use a longer text for summarization
long_text = """The quick brown fox jumps over the lazy dog. This is a very important sentence. \n\nIn a groundbreaking study published last week, scientists at the University of California, Berkeley, announced a significant breakthrough in renewable energy. Their research focuses on developing highly efficient solar cells that can convert sunlight into electricity with unprecedented accuracy. The new technology promises to reduce dependency on fossil fuels and significantly curb carbon emissions, addressing critical environmental concerns. \n\nThe lead researcher, Dr. Anya Sharma, stated in a press conference that 'this advancement could fundamentally change how we power our homes and industries, making clean energy accessible and affordable for everyone.' The team utilized novel perovskite materials in their design, which have shown remarkable stability and performance in initial tests. Funding for the project was provided by the National Science Foundation and several private investors committed to sustainable development. Further trials are planned for the next year to scale up production and test the cells in various real-world conditions. This discovery marks a pivotal moment in the global effort towards a greener future.\n\n""" + english_transcript # Adding our earlier transcript to make it longer

# Prepare the input for summarization. T5 models typically expect a prefix.
input_text = "summarize: " + long_text

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(device)

# Generate the summary
summary_ids = model.generate(
    inputs.input_ids,
    max_length=150, # Adjust max_length for the summary
    min_length=30,  # Adjust min_length for the summary
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"\nOriginal Text (partial):\n{long_text[:500]}...")
print(f"\nSummary:\n{summary}")

# For the proposed Chunking & Merging strategy for long texts:
# 1. Split text into chunks (e.g., 500-1000 words)
# 2. Summarize each chunk
# 3. Combine chunk summaries and summarize them again for final synthesis.
# This requires custom logic, which can be implemented as Python functions.

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

CUDA not available, running on CPU.

Original Text (partial):
The quick brown fox jumps over the lazy dog. This is a very important sentence. 

In a groundbreaking study published last week, scientists at the University of California, Berkeley, announced a significant breakthrough in renewable energy. Their research focuses on developing highly efficient solar cells that can convert sunlight into electricity with unprecedented accuracy. The new technology promises to reduce dependency on fossil fuels and significantly curb carbon emissions, addressing crit...

Summary:
scientists at the university of California, Berkeley, announced a significant breakthrough in renewable energy. their research focuses on developing highly efficient solar cells that convert sunlight into electricity with unprecedented accuracy. the new technology promises to reduce dependency on fossil fuels and significantly curb carbon emissions, addressing critical environmental concerns.


## 4. Speaker Diarization with NeMo Toolkit

Speaker diarization identifies 'who spoke when' in an audio recording, which is crucial for meeting minutes and attributing turns in a conversation. This section demonstrates speaker diarization using NVIDIA's NeMo toolkit.

To use `nemo_toolkit` for diarization, the process involves:
1.  **Creating a Manifest File**: A JSON file that describes the audio input for NeMo, specifying the audio file path and other metadata.
2.  **Downloading a Configuration**: A YAML file that contains the default parameters for the diarization process.
3.  **Loading and Updating Configuration**: Modifying the downloaded configuration to point to the correct manifest file, define the output directory, and specify pre-trained models such as `titanet_large` for speaker embeddings and `vad_multilingual_marblenet` for Voice Activity Detection (VAD).
4.  **Running the Diarizer**: Instantiating and executing the `ClusteringDiarizer` from `nemo.collections.asr.models` with the updated configuration to perform the diarization.
5.  **Printing Results**: Reading and displaying the diarization output, which is typically saved in RTTM format, showing the start time, duration, and assigned speaker for each segment in the audio.

I'll download another sample audio file for diarization.

In [9]:
# Install dependencies
!pip install --upgrade --force-reinstall wget
!apt-get update && apt-get install -y sox libsndfile1 ffmpeg
!pip install --upgrade --force-reinstall Cython
!pip install --upgrade --force-reinstall pytorch-lightning nemo_toolkit[asr]

  Using cached wget-3.2-py3-none-any.whl
  Attempting uninstall: wget
    Found existing installation: wget 3.2
    Uninstalling wget-3.2:
      Successfully uninstalled wget-3.2


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,803 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,309 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Get:13 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:14 h

  Using cached pytorch_lightning-2.6.1-py3-none-any.whl.metadata (21 kB)
  Using cached nemo_toolkit-2.7.2-py3-none-any.whl.metadata (77 kB)
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached lightning_utilities-0.15.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached cuda_bindings-13.2.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached fsspec-2024.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.meta

In [1]:
import os
import json
import wget
from omegaconf import OmegaConf
from nemo.collections.asr.models import ClusteringDiarizer

# 1. Create a simple manifest file for your audio
# NeMo expects a JSON line for each audio file
audio_file = "multi_speaker_audio.wav"
manifest_path = "input_manifest.json"

meta = {
    "audio_filepath": audio_file,
    "offset": 0,
    "duration": None,
    "label": "infer",
    "text": "-",
    "num_speakers": None, # Set this if you know the count (e.g., 2)
    "rttm_filepath": None
}

with open(manifest_path, 'w') as fp:
    json.dump(meta, fp)
    fp.write('\n')

# 2. Download the default configuration file
config_url = "https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/speaker_tasks/diarization/conf/inference/diar_infer_telephonic.yaml"
config_path = "diar_infer_telephonic.yaml"
if not os.path.exists(config_path):
    config_path = wget.download(config_url)

# 3. Load and update the configuration
config = OmegaConf.load(config_path)
config.diarizer.manifest_filepath = manifest_path
config.diarizer.out_dir = "nemo_outputs" # Where results are saved
config.diarizer.speaker_embeddings.model_path = "titanet_large" # Pre-trained model
config.diarizer.vad.model_path = "vad_multilingual_marblenet"

# 4. Run the Diarizer
sd_model = ClusteringDiarizer(cfg=config)
sd_model.diarize()

# 5. Print Results
# NeMo saves results in RTTM format in the 'nemo_outputs' folder
rttm_file = os.path.join("nemo_outputs", "pred_rttms", "multi_speaker_audio.rttm")
with open(rttm_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        start, duration, speaker = parts[3], parts[4], parts[7]
        print(f"Start: {start}s | Duration: {duration}s | Speaker: {speaker}")


[NeMo W 2026-03-28 08:19:58 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-03-28 08:20:01 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-03-28 08:20:01 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-03-28 08:20:01 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-03-28 08:20:01 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


[NeMo I 2026-03-28 08:20:07 clustering_diarizer:117] Loading pretrained vad_multilingual_marblenet model from NGC
[NeMo I 2026-03-28 08:20:07 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/vad_multilingual_marblenet/versions/1.10.0/files/vad_multilingual_marblenet.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo
[NeMo I 2026-03-28 08:20:07 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-03-28 08:20:08 classification_models:641] Please use the EncDecSpeakerLabelModel instead of this model. EncDecClassificationModel model is kept for backward compatibility with older models.
[NeMo W 2026-03-28 08:20:08 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/ami_train_0.63.json,/manifests/freesound_background_train.json,/manifests/freesound_laughter_train.json,/manifests/fisher_2004_background.json,/manifests/fisher_2004_speech_sampled.json,/manifests/google_train_manifest.json,/manifests/icsi_all_0.63.json,/manifests/musan_freesound_train.json,/manifests/musan_music_train.json,/manifests/musan_soundbible_train.json,/manifests/mandarin_train_sample.json,/manifests/german_train_sample.json,/manifests/spanish_train_sample.json,/manifests/french_train_sample.json,/manifests/russian_tr

[NeMo I 2026-03-28 08:20:09 save_restore_connector:285] Model EncDecClassificationModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2026-03-28 08:20:09 clustering_diarizer:150] Loading pretrained titanet_large model from NGC
[NeMo I 2026-03-28 08:20:09 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_large/versions/v1/files/titanet-l.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2026-03-28 08:20:10 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-03-28 08:20:10 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-03-28 08:20:10 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method 

...

[NeMo I 2026-03-28 08:29:49 clustering_diarizer:383] Saved embedding files to nemo_outputs/speaker_outputs/embeddings



[NeMo W 2026-03-28 08:29:49 speaker_utils:473] cuda=False, using CPU for eigen decomposition. This might slow down the clustering process.
clustering: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]

[NeMo I 2026-03-28 08:29:49 clustering_diarizer:451] Outputs are saved in /content/nemo_outputs directory



[NeMo W 2026-03-28 08:29:49 der:221] Check if each ground truth RTTMs were present in the provided manifest file. Skipping calculation of Diariazation Error Rate


Start: 2.300s | Duration: 0.430s | Speaker: speaker_1
Start: 6.620s | Duration: 0.670s | Speaker: speaker_0
Start: 7.500s | Duration: 0.875s | Speaker: speaker_1
Start: 8.375s | Duration: 1.500s | Speaker: speaker_0
Start: 9.875s | Duration: 1.000s | Speaker: speaker_1
Start: 10.875s | Duration: 3.250s | Speaker: speaker_0
Start: 14.125s | Duration: 4.250s | Speaker: speaker_1
Start: 18.375s | Duration: 3.250s | Speaker: speaker_0
Start: 21.625s | Duration: 6.500s | Speaker: speaker_1
Start: 28.125s | Duration: 1.875s | Speaker: speaker_0


## Next Steps for a Real-time Portal:

To build the full real-time, multi-user portal you described, you would typically need:

1.  **A Web Framework**: Such as Flask, FastAPI, or Django, to handle web requests and serve the UI.
2.  **Real-time Communication**: WebSockets would be essential for streaming audio from client microphones to the server and pushing translated/transcribed text back to clients in real-time.
3.  **Frontend**: HTML, CSS, JavaScript (potentially with a framework like React or Vue.js) to create the user interface for audio input, language selection, and displaying transcripts.
4.  **Audio Processing Backend**: A server-side component to receive audio chunks, feed them to the ASR model, send the transcript to the NMT model, perform diarization, and then send the translated text back to the appropriate client(s).
5.  **Multi-threading/Asynchronous Processing**: To handle multiple users concurrently without blocking.
6.  **Containerization**: Using Docker to package your application and deploy it to a cloud platform (e.g., Google Cloud Run, Kubernetes Engine).

These components demonstrated above are the *core AI/ML backend services* that your portal would call. Implementing the full portal would involve significant additional software development.